# CLOY Processing

This notebook is for you to work with the CLOY data folder.

In [2]:
import json, glob, re
from collections import Counter
import numpy as np
import pandas as pd

In [6]:
DATA_DIR = "/Users/annabel/Desktop/CS315/week4-tasks/data/CLOY"

## 1. Turn every CLOY thread into a vector of words

In [ ]:
# Helper function: 
# first sort all unique words in the submissions and comments alphabetically, then count the number of occurrences of each word in each document

def make_vectors(docs):
    counts = [Counter(re.findall(r"\w+", d.lower())) for d in docs]   # term frequency per document
    vocab = sorted(set(w for c in counts for w in c))                 # Step 1: vocabulary
    vectors = [[c[w] for w in vocab] for c in counts]                 # Step 2: one vector per document
    return vocab, vectors

In [ ]:
# Read jsonl files

def read_jsonl(pattern):
    rows = []
    for path in sorted(glob.glob(DATA_DIR + "/" + pattern)):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rows.append(json.loads(line))
    return rows

posts = read_jsonl("cloy_submissions_*.jsonl")
comments = read_jsonl("cloy_comments_*.jsonl")

print(len(posts), "posts,", len(comments), "comments")

3066 posts, 32057 comments


In [ ]:
# Combine posts and comments into threads

threads = {}
for p in posts:
    threads[p["post_id"]] = [p["title"], p["selftext"]]

for c in comments:
    post_id = c["post_id"][3:]           
    if post_id in threads:
        threads[post_id].append(c["comment_body"])

thread_ids = list(threads)
docs = []
for pid in thread_ids:
    texts = [t for t in threads[pid]] 
    docs.append(" ".join(texts))         

print(len(docs), "threads") 

3066 threads


In [12]:
# Create a matrix from the thread texts
vocab, vectors = make_vectors(docs)
df = pd.DataFrame(vectors, index=thread_ids, columns=vocab, dtype="int32")

print(len(vocab), "words in the vocabulary")
print(df.shape, "= (threads, words)")

30915 words in the vocabulary
(3066, 30915) = (threads, words)


## 2. Calculate Cosine Similarity

In [ ]:
# Helper function: 
# cosine similarity is defined as dot products / (length_i * length_j)

def cosine_matrix(vectors, ids):
    X = np.asarray(vectors, dtype="float32")
    lengths = np.sqrt((X * X).sum(axis=1))                 
    sims = (X @ X.T) / np.outer(lengths, lengths)          
    return pd.DataFrame(sims, index=ids, columns=ids)